# DFU Vision — fine-tune a backbone on your own labelled images (GPU)

This notebook trains a full transfer-learning model on a folder of DFU photographs organised **one sub-folder per class**, and exports it in the format the DFU Vision web app loads under *Train → Load Colab model*.

```
DFU_dataset/
  wagner_0/  img001.jpg ...
  wagner_1/
  ...
```

**Runtime → Change runtime type → GPU (T4)** before running. Free Colab gives a T4 for a few hours per day; Colab Pro removes most limits.

**Data governance.** Mount Drive only from an account that is permitted to hold the images (de-identified, or an institutional Google Workspace). Your DrPH protocol commits to an institution-hosted server under the PDPA 2010; for that case, upload a zip to the Colab session instead of mounting a personal Drive (see the second option below).

In [ ]:
#@title 1 · Install and import
!pip -q install tensorflowjs==4.22.0 tf_keras==2.16.0 scikit-learn
import os; os.environ["TF_USE_LEGACY_KERAS"] = "1"
import json, shutil, glob, time, numpy as np, tensorflow as tf, tf_keras as keras
from tf_keras import layers
print("TF", tf.__version__, "GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
#@title 2 · Point at the data (choose ONE)
SOURCE = "drive"   #@param ["drive", "zip_upload"]
DATA_DIR = "/content/drive/MyDrive/DFU_dataset"  #@param {type:"string"}

if SOURCE == "drive":
    from google.colab import drive; drive.mount("/content/drive")
else:
    from google.colab import files
    up = files.upload()                       # a zip containing one folder per class
    zname = list(up)[0]; shutil.unpack_archive(zname, "/content/data"); DATA_DIR = "/content/data"
    subs = [d for d in glob.glob(DATA_DIR + "/*") if os.path.isdir(d)]
    if len(subs) == 1 and any(os.path.isdir(x) for x in glob.glob(subs[0] + "/*")): DATA_DIR = subs[0]

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
CLASSES = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith("."))
counts = {c: len([f for f in os.listdir(os.path.join(DATA_DIR, c)) if f.lower().endswith(IMG_EXT)]) for c in CLASSES}
print("classes:", counts)
assert len(CLASSES) >= 2, "need at least two class folders"

In [ ]:
#@title 3 · Settings
BACKBONE = "EfficientNetB0"   #@param ["EfficientNetB0", "DenseNet121", "InceptionV3", "ResNet101V2", "MobileNetV2"]
IMG = 224          #@param {type:"integer"}
EPOCHS_HEAD = 10   #@param {type:"integer"}
EPOCHS_FT = 10     #@param {type:"integer"}
FT_LAYERS = 40     #@param {type:"integer"}
BATCH = 32
SCHEME = "auto"    #@param ["auto", "wagner", "texas", "custom"]
SEED = 42
# ResNet101 (v1) uses caffe-style BGR preprocessing that cannot be expressed as a Keras layer for TF.js export, so the V2 variant is offered instead.

In [ ]:
#@title 4 · Stratified 70 / 15 / 15 split and tf.data pipeline
from sklearn.model_selection import train_test_split
paths, labels = [], []
for i, c in enumerate(CLASSES):
    for f in sorted(os.listdir(os.path.join(DATA_DIR, c))):
        if f.lower().endswith(IMG_EXT): paths.append(os.path.join(DATA_DIR, c, f)); labels.append(i)
paths, labels = np.array(paths), np.array(labels)
p_tr, p_tmp, y_tr, y_tmp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
p_va, p_te, y_va, y_te = train_test_split(p_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print(f"train {len(p_tr)}  val {len(p_va)}  test {len(p_te)}")

def load(path, y):
    x = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    s = tf.minimum(tf.shape(x)[0], tf.shape(x)[1]); x = tf.image.resize_with_crop_or_pad(x, s, s)   # centre square crop
    x = tf.image.resize(x, [IMG, IMG]); return tf.cast(x, tf.float32), y                          # 0..255; preprocessing lives INSIDE the model
aug = keras.Sequential([layers.RandomFlip("horizontal_and_vertical"), layers.RandomRotation(0.25, fill_mode="reflect"),
                        layers.RandomZoom(0.2, fill_mode="reflect"), layers.RandomContrast(0.2), layers.RandomBrightness(0.15, value_range=(0, 255))])
def ds(p, y, train):
    d = tf.data.Dataset.from_tensor_slices((p, y)).map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if train: d = d.shuffle(2048, seed=SEED).batch(BATCH).map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    else: d = d.batch(BATCH)
    return d.prefetch(tf.data.AUTOTUNE)
d_tr, d_va, d_te = ds(p_tr, y_tr, True), ds(p_va, y_va, False), ds(p_te, y_te, False)
cw = {i: len(y_tr) / (len(CLASSES) * max(1, (y_tr == i).sum())) for i in range(len(CLASSES))}; print("class weights", cw)

In [ ]:
#@title 5 · Build the model (preprocessing baked in, Grad-CAM layer recorded)
inp = keras.Input((IMG, IMG, 3), name="image")
if BACKBONE == "EfficientNetB0":
    base = keras.applications.EfficientNetB0(input_tensor=inp, include_top=False, weights="imagenet"); GRAD = "top_activation"   # includes its own rescaling
elif BACKBONE == "DenseNet121":
    x = layers.Rescaling(1/255.)(inp); x = layers.Normalization(mean=[0.485, 0.456, 0.406], variance=[0.229**2, 0.224**2, 0.225**2])(x)
    base = keras.applications.DenseNet121(input_tensor=x, include_top=False, weights="imagenet"); GRAD = "relu"
elif BACKBONE == "InceptionV3":
    x = layers.Rescaling(1/127.5, offset=-1)(inp)
    base = keras.applications.InceptionV3(input_tensor=x, include_top=False, weights="imagenet"); GRAD = "mixed10"
elif BACKBONE == "ResNet101V2":
    x = layers.Rescaling(1/127.5, offset=-1)(inp)
    base = keras.applications.ResNet101V2(input_tensor=x, include_top=False, weights="imagenet"); GRAD = "post_relu"
else:
    x = layers.Rescaling(1/127.5, offset=-1)(inp)
    base = keras.applications.MobileNetV2(input_tensor=x, include_top=False, weights="imagenet"); GRAD = "out_relu"
base.trainable = False
h = layers.GlobalAveragePooling2D(name="gap")(base.get_layer(GRAD).output)
h = layers.Dropout(0.3, name="drop")(h)
out = layers.Dense(len(CLASSES), activation="softmax", name="head")(h)
model = keras.Model(inp, out, name=f"dfu_{BACKBONE.lower()}")
print(f"{BACKBONE}: {model.count_params():,} parameters, Grad-CAM layer = {GRAD}")

In [ ]:
#@title 6 · Train: new head first, then fine-tune the top of the backbone
def compile_(lr): model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)]
compile_(1e-3); h1 = model.fit(d_tr, validation_data=d_va, epochs=EPOCHS_HEAD, class_weight=cw, callbacks=cb)
base.trainable = True
for l in base.layers[:-FT_LAYERS]: l.trainable = False
for l in base.layers:
    if isinstance(l, layers.BatchNormalization): l.trainable = False
compile_(2e-5); h2 = model.fit(d_tr, validation_data=d_va, epochs=EPOCHS_FT, class_weight=cw, callbacks=cb)

In [ ]:
#@title 7 · Held-out test metrics (the ones named in the DrPH protocol)
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, matthews_corrcoef, f1_score
probs = model.predict(d_te, verbose=0); y_pred = probs.argmax(1)
print(classification_report(y_te, y_pred, target_names=CLASSES, digits=3))
cm = confusion_matrix(y_te, y_pred); print("confusion matrix (rows = true):\n", cm)
metrics = {"accuracy": float((y_pred == y_te).mean()), "macro_f1": float(f1_score(y_te, y_pred, average="macro")), "mcc": float(matthews_corrcoef(y_te, y_pred)), "n_test": int(len(y_te))}
try: metrics["auc"] = float(roc_auc_score(y_te, probs[:, 1]) if len(CLASSES) == 2 else roc_auc_score(y_te, probs, multi_class="ovr", average="macro"))
except Exception as e: print("AUC not computed:", e)
if len(CLASSES) == 2:
    tn, fp, fn, tp = cm.ravel(); metrics.update(sensitivity=tp/(tp+fn), specificity=tn/(tn+fp), ppv=tp/(tp+fp) if tp+fp else None, npv=tn/(tn+fn) if tn+fn else None)
print(json.dumps(metrics, indent=1))

In [ ]:
#@title 8 · Grad-CAM sanity check on a few test images
import matplotlib.pyplot as plt
conv_model = keras.Model(model.inputs, [model.get_layer(GRAD).output, model.output])
def gradcam(x, cls):
    with tf.GradientTape() as t:
        acts, p = conv_model(x[None]); s = p[0, cls]
    g = t.gradient(s, acts); w = tf.reduce_mean(g, axis=(1, 2), keepdims=True)
    m = tf.nn.relu(tf.reduce_sum(acts * w, -1))[0]; m = (m - tf.reduce_min(m)) / (tf.reduce_max(m) - tf.reduce_min(m) + 1e-6); return m.numpy()
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for k, (path, y) in enumerate(zip(p_te[:4], y_te[:4])):
    x, _ = load(path, y); pr = model.predict(x[None], verbose=0)[0]; c = pr.argmax()
    ax[0, k].imshow(x.numpy().astype("uint8")); ax[0, k].set_title(f"true {CLASSES[y]}"); ax[0, k].axis("off")
    ax[1, k].imshow(x.numpy().astype("uint8")); ax[1, k].imshow(tf.image.resize(gradcam(x, c)[..., None], [IMG, IMG])[..., 0], cmap="jet", alpha=0.45); ax[1, k].set_title(f"pred {CLASSES[c]} {pr[c]:.0%}"); ax[1, k].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
#@title 9 · Export for the web app (model.json + shards + classes.json) and save to Drive
import tensorflowjs as tfjs, re
OUT = "/content/dfu_tfjs_export"; shutil.rmtree(OUT, ignore_errors=True)
tfjs.converters.save_keras_model(model, OUT, quantization_dtype_map={"float16": "*"})
def guess_scheme(cs):
    if all(re.search(r"(?:wagner|grade|^w|^g)?[\s_\-]*[0-5](?![0-9])", c, re.I) for c in cs): return "wagner"
    if all(re.match(r"^[0-3]\s*[A-Da-d]$", c.strip()) for c in cs): return "texas"
    return "custom"
meta = {"name": f"{BACKBONE} fine-tuned ({len(CLASSES)} classes)", "backbone": BACKBONE, "classes": CLASSES, "gradcam_layer": GRAD, "input_range": "0-255",
        "scheme": SCHEME if SCHEME != "auto" else guess_scheme(CLASSES), "metrics": metrics, "trained": time.strftime("%Y-%m-%d"), "n_train": int(len(p_tr)), "n_val": int(len(p_va)), "n_test": int(len(p_te))}
json.dump(meta, open(OUT + "/classes.json", "w"), indent=1)
print(os.listdir(OUT), "\ntotal MB:", round(sum(os.path.getsize(os.path.join(OUT, f)) for f in os.listdir(OUT)) / 1e6, 1))
if SOURCE == "drive":
    dst = "/content/drive/MyDrive/DFU_model_export"; shutil.rmtree(dst, ignore_errors=True); shutil.copytree(OUT, dst); print("copied to", dst)
else:
    shutil.make_archive("/content/dfu_tfjs_export", "zip", OUT); from google.colab import files; files.download("/content/dfu_tfjs_export.zip")

## Using the exported model

1. Open DFU Vision → **Train on your data → Load Colab model**.
2. Select `model.json`, every `.bin` shard and `classes.json` from the export folder.
3. The model becomes active in **Assess a wound**; Grad-CAM uses the layer recorded in `classes.json`.

To ship it as the default model instead, copy the export into `dfu/model/` in the repository and adjust `index.html` (the demo model has two heads; a Colab export has one). Larger backbones (DenseNet121 ≈ 15 MB, InceptionV3 ≈ 45 MB at float16) load more slowly on a phone; EfficientNetB0 (≈ 9 MB) or MobileNetV2 (≈ 4.5 MB) are the practical choices for primary care.